In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)   # 显示所有列
pd.set_option('expand_frame_repr', False)   # 不允许换行
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_colwidth', None)
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')
train_data.info()  
test_data.info()
print('train_data中的缺失值：', train_data.isnull().sum())
print('test_data中的缺失值：', test_data.isnull().sum())
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].mean())
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].mean())
train_data['Cabin'] = train_data['Cabin'].fillna(method='bfill') # bfill表示后一个值
train_data['Cabin'] = train_data['Cabin'].fillna(method='ffill') # ffill表示前一个值
test_data['Cabin'] = test_data['Cabin'].fillna(method='bfill')
test_data['Cabin'] = test_data['Cabin'].fillna(method='ffill')
train_data['Embarked'] = train_data['Embarked'].fillna('S')
train_data['Embarked'].value_counts()
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].mean())
full_data = pd.concat([train_data, test_data], axis=0).reset_index()
full_data = full_data.drop('index', axis=1)

print('full_data中的缺失值：', full_data.isnull().sum())
Sex_mapDict = {'male' : 1, 'female' : 0}
full_data['Sex'] = full_data['Sex'].map(Sex_mapDict)
full_data['Sex'] = full_data['Sex'].map({'male':1, 'female':0})
Pclass_df = pd.DataFrame()
Pclass_df = pd.get_dummies(full_data['Pclass'], prefix='Pclass')
full_data = pd.concat([full_data, Pclass_df], axis=1)
full_data = full_data.drop('Pclass', axis=1)
Embarked_df = pd.DataFrame()
Embarked_df = pd.get_dummies(full_data['Embarked'], prefix='Embarked')
full_data = pd.concat([full_data, Embarked_df], axis=1)
full_data = full_data.drop('Embarked', axis=1)
def GetTitle(name):              # 编写一个新的函数
    str1 = name.split(',')[1]    # 用逗号进行分割，1代表取后半部分
    str2 = str1.split('.')[0]    # 用句号进行分割，0代表取前半部分
    str3 = str2.strip()          # strip()用于移除指定的符号，这里默认为空格  
    return str3

Name_df = pd.DataFrame()
Name_df['Title'] = full_data['Name'].map(GetTitle)  # 再次用上述的map函数，将GetTitle映射到列['Name']

#  这一步，像Mr、Miss的就是正确的头衔，而Mlle、Dona等可能要进行自我判断一下，这里把所有的值都统一到几个常见的头衔
Title_mapDict = {'Mr': 'Mr',
                 'Miss': 'Miss',
                 'Mrs': 'Mrs',
                 'Master': 'Master',
                 'Dr': 'Officer',
                 'Rev': 'Officer',
                 'Col': 'Officer',
                 'Ms': 'Mrs',
                 'Major': 'Officer',
                 'Mlle': 'Miss',
                 'Jonkheer': 'Mr',
                 'Don': 'Mr',
                 'Lady': 'Mrs',
                 'the Countess': 'Royalty',
                 'Dona': 'Royalty',
                 'Capt': 'Royalty',
                 'Sir': 'Officer',
                 'Mme': 'Royalty',
                 }

Name_df['Title'] = Name_df['Title'].map(Title_mapDict)
Name_df = pd.get_dummies(Name_df['Title'], prefix='Title')
full_data = pd.concat([full_data, Name_df], axis=1)
full_data = full_data.drop('Name', axis=1)
full_data['Cabin'] = full_data['Cabin'].astype(str)            # 首先将这列转换为字符串类型
full_data['Cabin'] = full_data['Cabin'].map(lambda x : x[0])   # 接着用lambda函数实现提取首字符

Cabin_df = pd.DataFrame()
Cabin_df = pd.get_dummies(full_data['Cabin'], prefix='Cabin')

full_data = pd.concat([full_data, Cabin_df], axis=1)
full_data = full_data.drop('Cabin', axis=1)
Family_df = pd.DataFrame()
Family_df['Family'] = full_data['Parch'] + full_data['SibSp'] + 1
Family_df['Family_S'] = Family_df['Family'].map(lambda x : 1 if x == 1 else 0)
Family_df['Family_M'] = Family_df['Family'].map(lambda x : 1 if 2 <= x <= 3 else 0)
Family_df['Family_L'] = Family_df['Family'].map(lambda x : 1 if 4 < x < 5 else 0)
Family_df['Family_Xl'] = Family_df['Family'].map(lambda x : 1 if 5 < x else 0)


Family_df = Family_df.iloc[:, 1:]
full_data = pd.concat([full_data, Family_df], axis=1)
Fare_df = pd.DataFrame()
Fare_df['Fare'] = full_data['Fare']
Fare_df['Fare_Low'] = Fare_df['Fare'].map(lambda x : 1 if 0 <= x < 50 else 0)
Fare_df['Fare_Medium'] = Fare_df['Fare'].map(lambda x : 1 if 50 <= x < 100 else 0)
Fare_df['Fare_High'] = Fare_df['Fare'].map(lambda x : 1 if 100 < x else 0)

Fare_df = Fare_df.iloc[:, 1:]

full_data = pd.concat([full_data, Fare_df], axis=1)
full_data = full_data.drop('Fare', axis=1)
full_X = full_data.drop('PassengerId',axis=1)
full_X = full_X.drop('SibSp', axis=1)
full_X = full_X.drop('Parch', axis=1)
full_X = full_X.drop('Ticket', axis=1)
Row = 890
train_X = full_X.loc[0:Row, :]                 # 前891行，即原train数据集行数
train_X = train_X.drop('Survived', axis=1)     # 剔除'Survived'列
train_y = full_X.loc[0:Row, 'Survived']        # 原始train数据集行数
model = LogisticRegression()
    
try:
    # 训练模型
    model.fit(train_X, train_y)
    # 评估模型在训练集上的表现
    acc = model.score(train_X, train_y)
    print('逻辑回归模型评价:', acc)
    print("train_X缺失值情况：", train_X.isnull().any().any())
    print("train_y缺失值情况：", train_y.isnull().any())
    model = LogisticRegression(solver='liblinear')
    pred_df.to_csv('/kaggle/working/result.csv', index=False)

    # 预测并转换结果类型
    pred_Y = model.predict(pred_X)
    pred_Y = pred_Y.astype(int)

    # 后续处理，假设full_data、passenger_id相关操作正确
    passenger_id = full_data.loc[Row+1:, 'PassengerId']
    pred_df = pd.DataFrame({'PassengerId': passenger_id, 'Survived': pred_Y}).reset_index()
    pred_df.shape
    pred_df.head()
    pred_df.to_csv('/kaggle/titanic/pred.csv',index=False)
    # 保存结果
except Exception as e:
    print(f"代码运行出错，错误信息：{e}")
pred_X = full_X.loc[Row+1:, :].reset_index()      # 取原test数据集的行数
pred_X = pred_X.drop('index', axis=1)             # 去掉index与'Survived'列
pred_X = pred_X.drop('Survived', axis=1)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. 处理缺失值
imputer = SimpleImputer(strategy='mean')
train_X_filled = imputer.fit_transform(train_X)

# 2. 训练模型
model = LogisticRegression(max_iter=1000)
model.fit(train_X_filled, train_y)

# 3. 预测并保存结果
try:
    pred_X_filled = imputer.transform(pred_X)
    pred_Y = model.predict(pred_X_filled).astype(int)
    
    passenger_id = full_data.loc[Row+1:, 'PassengerId']
    pred_df = pd.DataFrame({'PassengerId': passenger_id, 'Survived': pred_Y})
    pred_df.to_csv('/kaggle/working/submission.csv', index=False)
    print("结果保存成功！")

except Exception as e:
    print(f"错误发生：{e}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass  